In [1]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.data as Data
import torchvision
import torchvision.transforms as transforms

# ==========================================
# 1. 超参数与环境配置
# ==========================================
# 自动检测 GPU (CUDA)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"当前使用的计算设备: {device}")

learning_rate = 1e-4
# PyTorch 的 Dropout 参数代表的是“丢弃率” (1 - keep_prob)
dropout_rate = 0.3
max_epoch = 3
BATCH_SIZE = 50
DOWNLOAD_MNIST = False

if not (os.path.exists("./mnist/")) or not os.listdir("./mnist/"):
    DOWNLOAD_MNIST = True

# ==========================================
# 2. 数据加载管道 (Data Pipeline)
# ==========================================
# 训练集
train_data = torchvision.datasets.MNIST(
    root="./mnist/",
    train=True,
    transform=transforms.ToTensor(),  # 自动将 PIL Image 转为 Tensor 并归一化到 [0.0, 1.0]
    download=DOWNLOAD_MNIST,
)
train_loader = Data.DataLoader(dataset=train_data, batch_size=BATCH_SIZE, shuffle=True)

# 测试集 (同样使用 DataLoader 管理)
test_data = torchvision.datasets.MNIST(
    root="./mnist/", train=False, transform=transforms.ToTensor()
)
# 为了加快测试速度，测试集可以设置更大的 batch_size，且不需要 shuffle
test_loader = Data.DataLoader(dataset=test_data, batch_size=1000, shuffle=False)


# ==========================================
# 3. 模型定义 (Model Architecture)
# ==========================================
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        # Block 1: 1x28x28 -> 32x14x14
        self.conv1 = nn.Sequential(
            nn.Conv2d(
                in_channels=1, out_channels=32, kernel_size=7, stride=1, padding=3
            ),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
        )
        # Block 2: 32x14x14 -> 64x7x7
        self.conv2 = nn.Sequential(
            nn.Conv2d(
                in_channels=32, out_channels=64, kernel_size=5, stride=1, padding=2
            ),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
        )
        # 展平后的特征数: 64 个通道，每个通道 7x7 大小
        self.out1 = nn.Linear(64 * 7 * 7, 1024, bias=True)
        self.dropout = nn.Dropout(p=dropout_rate)
        self.out2 = nn.Linear(1024, 10, bias=True)

    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        # 动态获取 batch_size 并展平剩余维度
        x = x.view(x.size(0), -1)
        out1 = F.relu(self.out1(x))
        out1 = self.dropout(out1)
        out2 = self.out2(out1)
        # 返回未激活的 Logits，配合 CrossEntropyLoss 使用
        return out2


# ==========================================
# 4. 评估与训练逻辑 (Training & Evaluation)
# ==========================================
def evaluate_accuracy(model, data_loader, device):
    """在给定数据集上评估模型准确率"""
    model.eval()  # 切换到评估模式 (关闭 Dropout 等)
    correct = 0
    total = 0

    with torch.no_grad():  # 禁用梯度计算，节省显存并加速
        for x, y in data_loader:
            x, y = x.to(device), y.to(device)
            outputs = model(x)
            # 获取最大概率的类别索引
            _, predicted = torch.max(outputs.data, 1)
            total += y.size(0)
            correct += (predicted == y).sum().item()

    return correct / total


def train():
    # 实例化模型并推送到设备
    model = CNN().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    # CrossEntropyLoss 内部自带 LogSoftmax，要求输入未经 Softmax 的 Logits
    loss_func = nn.CrossEntropyLoss()

    for epoch in range(max_epoch):
        model.train()  # 每个 epoch 开始时确保切换回训练模式

        for step, (x, y) in enumerate(train_loader):
            # 将数据推送到设备
            x, y = x.to(device), y.to(device)

            output = model(x)
            loss = loss_func(output, y)

            optimizer.zero_grad()  # 清空过往梯度
            loss.backward()  # 反向传播计算当前梯度
            optimizer.step()  # 更新网络参数

            if step != 0 and step % 100 == 0:
                # 打印日志时，计算整个测试集的准确率
                test_acc = evaluate_accuracy(model, test_loader, device)
                print(
                    f"Epoch: {epoch+1}/{max_epoch} | Step: {step} | Train Loss: {loss.item():.4f} | Test Accuracy: {test_acc:.4f}"
                )


if __name__ == "__main__":
    train()

当前使用的计算设备: cuda
Failed to download (trying next):
HTTP Error 404: Not Found



100.0%


Extracting ./mnist/MNIST\raw\train-images-idx3-ubyte.gz to ./mnist/MNIST\raw

Failed to download (trying next):
HTTP Error 404: Not Found



100.0%


Extracting ./mnist/MNIST\raw\train-labels-idx1-ubyte.gz to ./mnist/MNIST\raw

Failed to download (trying next):
HTTP Error 404: Not Found



100.0%


Extracting ./mnist/MNIST\raw\t10k-images-idx3-ubyte.gz to ./mnist/MNIST\raw

Failed to download (trying next):
HTTP Error 404: Not Found



100.0%


Extracting ./mnist/MNIST\raw\t10k-labels-idx1-ubyte.gz to ./mnist/MNIST\raw

Epoch: 1/3 | Step: 100 | Train Loss: 0.6161 | Test Accuracy: 0.8674
Epoch: 1/3 | Step: 200 | Train Loss: 0.5347 | Test Accuracy: 0.9042
Epoch: 1/3 | Step: 300 | Train Loss: 0.4069 | Test Accuracy: 0.9140
Epoch: 1/3 | Step: 400 | Train Loss: 0.2693 | Test Accuracy: 0.9335
Epoch: 1/3 | Step: 500 | Train Loss: 0.1498 | Test Accuracy: 0.9476
Epoch: 1/3 | Step: 600 | Train Loss: 0.3097 | Test Accuracy: 0.9506
Epoch: 1/3 | Step: 700 | Train Loss: 0.1678 | Test Accuracy: 0.9590
Epoch: 1/3 | Step: 800 | Train Loss: 0.0582 | Test Accuracy: 0.9622
Epoch: 1/3 | Step: 900 | Train Loss: 0.0354 | Test Accuracy: 0.9679
Epoch: 1/3 | Step: 1000 | Train Loss: 0.0904 | Test Accuracy: 0.9690
Epoch: 1/3 | Step: 1100 | Train Loss: 0.0668 | Test Accuracy: 0.9720
Epoch: 2/3 | Step: 100 | Train Loss: 0.1982 | Test Accuracy: 0.9748
Epoch: 2/3 | Step: 200 | Train Loss: 0.0850 | Test Accuracy: 0.9715
Epoch: 2/3 | Step: 300 | Train Loss: 